# LWLM: Large Wireless Localization Model — End-to-End Pipeline

Reproducible, restructured implementation of
**"Large Wireless Localization Model (LWLM): A Foundation Model for Positioning in 6G Networks"**
(Pan et al., arXiv:2505.10134) — official repo: [guangjinpan/LWLM](https://github.com/guangjinpan/LWLM).

**Pipeline overview**

```
Dataset/*.mat (DeepMIMO O1_3p5, YOUR files)
        │  Step 1 — DeepMIMOv3 conversion (one-time)
        ▼
processed/O1_3p5/{train,test}/BSx_128sc_32at_yM/x_i.h5py  + index .npy splits
        │  Step 3 — SSL pretraining  (SF-MCM + DTI + PICL)
        ▼
checkpoints/pretrain_mix/*.ckpt
        │  Step 4-6 — task-adaptive fine-tuning
        ▼
SingleBSLoc │ MultiBSLoc │ ToA │ AoA   →   Step 7 — test set metrics + error CDFs (results/)
```

The model, data reading, transformations and losses are **unchanged** from the
official repository; the refactor only fixes broken paths, missing files and
environment issues so that this notebook runs top-to-bottom.

## 0 · Setup

Imports, project-root detection and reproducibility. Run this cell first — it
adds the project root to `sys.path` so the `lwlm` package can be imported no
matter where Jupyter was launched.

In [ ]:
import os, sys, gc, json, time, random
from pathlib import Path

# --- locate the project root (folder containing the lwlm/ package) ---------
def find_project_root() -> Path:
    env = os.environ.get("LWLM_ROOT")
    if env and (Path(env) / "lwlm").is_dir():
        return Path(env).resolve()
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "lwlm" / "paths.py").exists():
            return cand.resolve()
    raise FileNotFoundError(
        "Could not locate the lwlm package. Launch Jupyter from the LWLM "
        "project folder or set the LWLM_ROOT environment variable.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")  # as in the original train.py

import numpy as np
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"project root : {PROJECT_ROOT}")
print(f"PyTorch      : {torch.__version__}")
print(f"device       : {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == 'cuda' else ""))

### Configuration

Defaults follow the **paper scale** used by the original repository's README
(`FT_dataset=10000`, `BS_Num=4`, 4 BSs × 3 bandwidths for pretraining, ...).

> ⚡ Set `SMALL_DEBUG = True` for a fast smoke run (1 epoch, 200 samples) — it
> still uses **your real data**, just less of it, to verify the plumbing before
> launching the multi-hour paper-scale training.

In [ ]:
from lwlm.config import paper_envpara, apply_overrides
from lwlm import paths

SMALL_DEBUG = False   # ← flip to True for a quick smoke run

if SMALL_DEBUG:
    CFG = dict(pretrain_epochs=1, ft_epochs=2,
               ft_dataset=200, test_samples=100,
               batch_size=8,
               num_workers=0)                 # 0 = safest inside Jupyter (Windows spawn)
else:
    CFG = dict(pretrain_epochs=140,           # original README ships a 'testepoch=139' ckpt
               ft_epochs=1000,                # original train.py default
               ft_dataset=10000,              # paper fine-tuning size
               test_samples=5400,             # original test-pool size (auto-capped)
               batch_size=32,
               num_workers=0 if os.name == "nt" else 4)

print("run configuration:", CFG)

def envpara_for(task, bs_num, epochs, **overrides):
    """Build an EnvPara dict (same keys as the original train.py / test_*.py)."""
    ep = paper_envpara(task=task, bs_num=bs_num, epochs=epochs,
                       ft_dataset=CFG["ft_dataset"],
                       batch_size=CFG["batch_size"],
                       num_workers=CFG["num_workers"],
                       device=DEVICE, seed=SEED)
    ep["model_path"] = str(paths.checkpoints_dir() / task)
    return apply_overrides(ep, **overrides)

## 1 · Data preparation — your `Dataset/` .mat files → per-sample `.h5py`

The model consumes one `.h5py` file per (UE, BS, bandwidth) with the **exact**
format of the original repo (`channel_real, channel_imag, UElocation, LoS,
distance, BSlocation, DoD_phi, DoD_theta, bandwidth`). This step converts your
DeepMIMO **O1_3p5** scenario accordingly — no dataset is generated here from
scratch; your `Dataset/` folder is the only source.

* train pool → BS 3/4/9/10 × BW 10/20/50 MHz × `user_rows` grid rows (200 UE/row)
* test pool  → BW 10 MHz × 5400 UE positions × BS 3/4/9/10 (paper test config)
* index splits `train.npy / val_1000.npy / pretrain_data.npy` are written automatically
  *(the original repo referenced these files but never created them — the #1 crash)*

**Costs (paper scale, `user_rows=100` → 20 000 UE):** ≈ 240k small files ≈ 4–6 GB,
a few hours of ray-tracing, one time only. The cell **skips itself** when the
index files already exist (delete `processed/` to regenerate).

In [ ]:
from lwlm.data_generation import generate_all

# ---- sanity check: is the raw scenario present? ----------------------------
scen_dir = paths.dataset_dir()
mat_locations = [scen_dir / "O1_3p5"] + [p for p in scen_dir.iterdir() if p.is_dir()] \
    if scen_dir.exists() else []
n_mat = len(list(scen_dir.glob("*.mat"))) + sum(len(list(p.glob("*.mat"))) for p in mat_locations) \
    if scen_dir.exists() else 0
print(f"Dataset folder : {scen_dir}")
print(f".mat files found: {n_mat}")
if n_mat == 0 and not (paths.processed_dir() / "train.npy").exists():
    raise FileNotFoundError(
        f"No DeepMIMO .mat files under {scen_dir}.\n"
        "Download the O1_3p5 scenario from https://deepmimo.net/scenarios/o1-scenario/\n"
        "and extract it so that either Dataset/O1_3p5/*.mat or Dataset/*.mat exists.")

# ---- convert (auto-skips if already done) ----------------------------------
# user_rows: number of O1_3p5 grid rows for the train pool. 100 rows = 20k UE
# positions -> 10k fine-tune + 1k val + 9k pretraining pool (paper-scale FT).
# Raise it (e.g. 2700 = full grid) for the full 5.8M-sample paper pretraining set.
USER_ROWS = 100

meta = generate_all(user_rows=USER_ROWS,
                    n_finetune=200 if SMALL_DEBUG else CFG["ft_dataset"],
                    n_val=1000,
                    skip_existing=True)

CFG["test_samples"] = min(CFG["test_samples"], int(meta.get("test", {}).get("n_test", 5400)))
print("test samples used:", CFG["test_samples"])

## 2 · Inspect one CSI sample

Each raw sample is a complex channel tensor of shape **(1 UE antenna, 32 BS
antennas, 128 subcarriers)** — $H \in \mathbb{C}^{32\times128}$ after dropping the
single-antenna axis, exactly as DeepMIMOv3 emits it. The dataloader (unchanged
from the original) turns it into three 2-map views:

| view | maps | used by |
|------|------|---------|
| spatial-frequency, amplitude+phase | $\|H\|, \angle H$ | all task decoders |
| spatial-frequency, real+imag | $\Re H, \Im H$ | (pretraining input) |
| delay–angle, real+imag | IFFT/FFT of $\Re H + j\Im H$ | DTI pretraining loss |

Below we plot a raw sample exactly as the model would see it (after the
original `channel_normalization`).

In [ ]:
import h5py
from lwlm.transforms import channel_normalization, freq_to_delay_angle_domain

bs_folder = paths.processed_split_dir("train") / "BS3_128sc_32at_10M"
sample_id = 0
with h5py.File(bs_folder / f"3_{sample_id}.h5py", "r") as f:
    H = f["channel_real"][:] + 1j * f["channel_imag"][:]
    ue_loc = f["UElocation"][:]
    bs_loc = f["BSlocation"][:]
    dist = float(f["distance"][()])
    dod_phi = f["DoD_phi"][:]
    print("raw H shape (UE ant, BS ant, subcarrier):", H.shape)
    print("UE location [m] :", ue_loc)
    print("BS location [m] :", bs_loc)
    print("distance [m]    :", dist)
    print("DoD phi [deg]   :", dod_phi.ravel()[:3], "...")

Hn = channel_normalization(H)                      # original normalization (raw 3-D input)
H2 = Hn[0] if Hn.ndim == 3 else Hn                 # drop the single-UE-antenna axis -> (32, 128)
H_aa = np.zeros((2, H2.shape[0], H2.shape[1]))
H_aa[0] = np.abs(H2); H_aa[1] = np.angle(H2)       # amplitude / phase view
H_da = freq_to_delay_angle_domain(H_aa)            # delay-angle view (original transform)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4), constrained_layout=True)
im0 = axes[0].imshow(20*np.log10(H_aa[0] + 1e-9), aspect="auto", cmap="viridis")
axes[0].set_title("|H|  [dB]  (subcarrier x antenna)"); axes[0].set_xlabel("antenna"); axes[0].set_ylabel("subcarrier")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(H_aa[1], aspect="auto", cmap="twilight")
axes[1].set_title("phase  (subcarrier x antenna)"); axes[1].set_xlabel("antenna"); axes[1].set_ylabel("subcarrier")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(20*np.log10(np.abs(H_da) + 1e-9), aspect="auto", cmap="viridis")
axes[2].set_title("delay-angle domain |H| [dB]"); axes[2].set_xlabel("delay tap"); axes[2].set_ylabel("angle bin")
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.show()

## 3 · Self-supervised pretraining — `pretrain_mix` (SF-MCM + DTI + PICL)

The hybrid SSL objective jointly optimises (paper Sec. III):

* **SF-MCM** — spatial-frequency masked channel modelling: mask 16 antennas + 64
  subcarriers, reconstruct the patched input (MSE);
* **DTI** — domain-transformation invariance: reconstruct the real+imag map from
  the amplitude+phase view (complex cosine loss);
* **PICL** — position-invariant contrastive learning: NT-Xent loss pulling
  together channels of the *same UE location* seen from *different BS/BW
  configurations* (the dataloader samples 2 such views per item).

The dataloader randomly pairs every sample with a second (BS, BW) view — this
is exactly the original `generate_Dataset` pretrain branch.

In [ ]:
from lwlm.training import train
from lwlm.wrapper import Wrapper

# ---------------- pretraining configuration (paper scale) -------------------
ep_pre = envpara_for("pretrain_mix", bs_num=1, epochs=CFG["pretrain_epochs"],
                     is_load=0, is_frozen=0,
                     mask_antenna_number=16, mask_subcarrier_number=64)

best_ckpt = {}
best_ckpt["pretrain_mix"] = train(ep_pre)
print("pretrained model:", best_ckpt["pretrain_mix"])

## 4 · Fine-tune — Single-BS localization (`SingleBSLoc`)

The frozen-or-not encoder is fed the pilot-masked CSI (with
`pilot_*_interval = 1` the full channel is visible) together with the BS
configuration embedding; a lightweight MLP head (`SB_positioninglayer`)
regresses the 2-D UE position. Loss = mean Euclidean distance. The encoder
weights are initialised from the pretraining checkpoint (`is_load=1`).

In [ ]:
assert "pretrain_mix" in best_ckpt and best_ckpt["pretrain_mix"], \
    "Run Step 3 first (or point load_pretrained_mdl_path at an existing ckpt)."

ep_sb = envpara_for("SingleBSLoc", bs_num=4, epochs=CFG["ft_epochs"],
                    is_load=1, is_frozen=0,
                    load_pretrained_mdl_path=best_ckpt["pretrain_mix"],
                    BW=10, pilot_subcarrier_interval=1, pilot_antenna_interval=1)

best_ckpt["SingleBSLoc"] = train(ep_sb)

## 5 · Fine-tune — Multi-BS fusion localization (`MultiBSLoc`)

Each of the 4 BSs produces a per-BS position estimate through a shared encoder
and per-BS heads; an **attention pooling** module (`AttentionPooling`) fuses the
per-BS estimates into the final position — this supports an arbitrary number of
BSs at inference (paper Sec. IV-B). Training randomly samples T ∈ {2,3,4} BSs
per forward pass (original `multiBSLoc_att`, `mode="train"`).

In [ ]:
ep_mb = envpara_for("MultiBSLoc", bs_num=4, epochs=CFG["ft_epochs"],
                    is_load=1, is_frozen=0,
                    load_pretrained_mdl_path=best_ckpt["pretrain_mix"],
                    BW=10, pilot_subcarrier_interval=1, pilot_antenna_interval=1)

best_ckpt["MultiBSLoc"] = train(ep_mb)

## 6 · Fine-tune — ToA estimation (`toa`) and AoA estimation (`aoa`)

Same encoder, single-BS setup (BS_Num=1), with lightweight regression heads:
* **ToA** → UE–BS distance (`toa_layer`, L1 loss against `distance/100`),
* **AoA** → DoD azimuth angle (`aoa_layer`, L1 loss against `DoD_phi/100`).

In [ ]:
for task in ["toa", "aoa"]:
    ep = envpara_for(task, bs_num=1, epochs=CFG["ft_epochs"],
                     is_load=1, is_frozen=0,
                     load_pretrained_mdl_path=best_ckpt["pretrain_mix"],
                     BW=10, pilot_subcarrier_interval=1, pilot_antenna_interval=1)
    best_ckpt[task] = train(ep)

print(json.dumps({k: str(v) for k, v in best_ckpt.items()}, indent=2))

## 7 · Test & evaluation

Runs the original test loop on the 5400-UE test pool (BS 3 — single-BS tasks;
BS 3/4/9/10 — multi-BS fusion), de-normalises predictions exactly as the
original scripts (`×10` positions, `×100` distance/angle), reports
mean / median / P90 errors, saves `.txt` result files and paper-style **CDF
plots** into `results/`.

In [ ]:
from lwlm.evaluation import (run_inference, summarize_errors,
                             save_results_txt, plot_error_cdf)

eval_configs = [
    ("SingleBSLoc", 1, "Localization error [m]"),
    ("MultiBSLoc",  4, "Localization error [m]"),
    ("toa",         1, "ToA / distance error [m]"),
    ("aoa",         1, "AoA error [deg]"),
]

summary = {}
for task, bs_num, xlabel in eval_configs:
    print(f"\n=== testing {task} (BS_Num={bs_num}) ===")
    ep = envpara_for(task, bs_num=bs_num, epochs=1, test_samples=CFG["test_samples"])
    ckpt = best_ckpt.get(task)
    if not ckpt or not Path(str(ckpt)).exists():
        print(f"[skip] no checkpoint for {task} - run the fine-tuning step above first")
        continue
    pred, label, losses = run_inference(Wrapper, ep, str(ckpt))
    errors = summarize_errors(pred, label, task)
    save_results_txt(pred, label, f"{task}_test")
    plot_error_cdf(errors, f"LWLM — {task} error CDF (O1_3p5, BW 10 MHz)",
                   f"{task}_test", xlabel=xlabel)
    summary[task] = float(np.mean(errors))

print("\n================ summary (mean test error) ================")
for k, v in summary.items():
    print(f"  {k:<12}: {v:.3f}")

## Summary & command-line reference

Everything above can also be run from the terminal (same CLI arguments as the
original repository):

```bash
python scripts/generate_data.py --user_rows 100        # Step 1
python scripts/train.py --task pretrain_mix \
    --model_path checkpoints/pretrain_mix --is_frozen 0 --is_load 0 --depth 4
python scripts/train.py --task SingleBSLoc \
    --model_path checkpoints/SingleBSLoc --is_frozen 0 --is_load 1 --depth 4 \
    --load_pretrained_mdl_path <pretrain_ckpt> \
    --FT_dataset 10000 --BW 10 \
    --pilot_subcarrier_interval 1 --pilot_antenna_interval 1 --BS_Num 4
python scripts/test_singleBS.py --ckpt <singleBS_ckpt>
python scripts/test_multiBS.py  --ckpt <multiBS_ckpt>
python scripts/test_toa.py      --ckpt <toa_ckpt>
python scripts/test_aoa.py      --ckpt <aoa_ckpt>
```

Artifacts produced by this notebook:

| path | content |
|------|---------|
| `processed/` | h5py pool + `train.npy` / `val_1000.npy` / `pretrain_data.npy` |
| `checkpoints/<task>/testepoch=*.ckpt` | best checkpoint per task (monitors `val/ave_loss`) |
| `checkpoints/<task>/test_latest.ckpt` | most recent weights |
| `results/*.txt` | per-UE `label pred` outputs |
| `results/*_cdf.png` | error CDF plots |
| `logs/<task>/` | TensorBoard scalars (`tensorboard --logdir logs`) |

**Notes for reproducing the paper numbers**

* paper pretraining uses the *full* O1_3p5 grid (5.86M channel samples); here
  `USER_ROWS=100` (20k UE × 12 configs) is the practical paper-scale starting
  point — raise `USER_ROWS` in Step 1 for the full set;
* checkpoints are selected by validation loss, matching the original
  `ModelCheckpoint(monitor="val/ave_loss")` configuration;
* ToA/AoA/single-BS tests use BS 3 only (as in the original test scripts).

If you use this code, please cite the original paper:

```bibtex
@article{pan2025large,
  title={Large Wireless Localization Model (LWLM): A Foundation Model for Positioning in 6G Networks},
  author={Pan, Guangjin and Huang, Kaixuan and Chen, Hui and Zhang, Shunqing and H{\"a}ger, Christian and Wymeersch, Henk},
  journal={arXiv preprint arXiv:2505.10134},
  year={2025}
}
```